# real_groundtruth_downscale.ipynb

Real-data analogue of the paper's controlled synthetic-detector experiment
(Esri World Imagery, Table I): same structure, but using only real
Sentinel-2 data. All function definitions live in
`tests/real_groundtruth_common_tools.py` (imported below as `gt`) -- this
notebook is deliberately just calls and inline inspection of intermediate
results, step by step.

**Protocol** (see the module's own docstring for the full rationale and
revision history):

1. fetch a real, native 10 m Sentinel-2 B04 UTM patch
2. PSF-aware resample the real (undegraded) data to `gt.NATIVE_LEVEL` (20)
3. plain (no-PSF) NESTED downgrade to `gt.REF_LEVEL` (18) -- **the reference**
4. Gaussian-blur the real data to the coarse effective resolution
5. point-sample (decimate) every `gt.BLOCK`-th pixel (not a block average)
6. PSF-aware resample the coarse samples directly onto `gt.REF_LEVEL`
7. compare against the reference, cell-for-cell, at `gt.REF_LEVEL`

**Changing parameters**: edit the constants in `real_groundtruth_common_tools.py`,
or override them from here before calling a function, e.g. `gt.REF_LEVEL = 17`
-- both work because every function reads its configuration from the
module's own globals at call time (see the module docstring for why this
requires `import ... as gt`, not `from ... import *`).

## Setup

In [ ]:
import sys
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start=None, marker="healpix_resample"):
    """Walk upward from `start` (default: this notebook's cwd) until a
    directory containing `marker` (the package's own source tree) is found.
    More robust than a hardcoded `..` -- works regardless of whether the
    notebook is actually run with cwd=notebooks/ or something else, and
    sidesteps any ambiguity about where the repo root is."""
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the repo root (looked for a '{marker}' directory above {p}).")


REPO_ROOT = find_repo_root()
MODULE_PATH = REPO_ROOT / "tests" / "real_groundtruth_common_tools.py"
print("repo root:  ", REPO_ROOT)
print("module path:", MODULE_PATH, "(exists:", MODULE_PATH.exists(), ")")

# Loaded directly from its file path (importlib.util), NOT via
# `import tests.real_groundtruth_common_tools`: a plain dotted import
# depends on `tests` resolving, on sys.path, to *this* directory -- but
# `tests` is a common, generic top-level name, and if anything else on
# sys.path (another package, another checkout of this repo, a stray
# editable-install path) also happens to expose a `tests` package or
# namespace, Python can silently bind `tests` to that other one instead,
# producing exactly "No module named 'tests.real_groundtruth_common_tools'"
# (tests found, submodule not found in it) rather than a clean, obvious
# "tests not found". Loading straight from MODULE_PATH sidesteps that
# entirely: there is no package-name resolution involved, so no ambiguity.
spec = importlib.util.spec_from_file_location("real_groundtruth_common_tools", MODULE_PATH)
gt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gt)

gt.describe_config()


## Step 1 -- fetch real native 10 m patches

Shared cache with `test-resample-paper.ipynb` (`notebooks/data/{scene}_data.zarr`) -- reused if already present.

In [ ]:
for scene in gt.benchmark_coordinates:
    gt.extract_bench_data(scene)


## Steps 2-3 -- build the reference (real data, no degradation)

Inspect: number of retained cells and value range at `NATIVE_LEVEL`, then after the downgrade to `REF_LEVEL`.

In [ ]:
refs = {scene: gt.build_reference(scene) for scene in gt.benchmark_coordinates}

for scene, ref in refs.items():
    n_native = len(ref["cell_ids_native"])
    n_ref = len(ref["cell_ids_ref"])
    print(f"{scene:12s}  level {gt.NATIVE_LEVEL}: {n_native:7d} cells "
          f"[{ref['cell_data_native'].min():.4f}, {ref['cell_data_native'].max():.4f}]"
          f"   ->  level {gt.REF_LEVEL}: {n_ref:6d} cells "
          f"[{ref['cell_data_ref'].min():.4f}, {ref['cell_data_ref'].max():.4f}]")


Quick look: histogram of the reference values for one scene.

In [ ]:
scene = "urban"
plt.figure(figsize=(5, 3))
plt.hist(refs[scene]["cell_data_ref"], bins=60)
plt.title(f"{scene}: reference values at level {gt.REF_LEVEL}")
plt.xlabel("reflectance")
plt.show()


## Steps 4-5 -- degrade to coarse UTM samples

Inspect: coarse image shape and value range per scene, then a quick side-by-side of native vs. degraded for one scene.

In [ ]:
coarse = {scene: gt.build_coarse_grid(scene) for scene in gt.benchmark_coordinates}

for scene, g in coarse.items():
    print(f"{scene:12s}  native {g['img_native'].shape}  ->  coarse {g['img_coarse'].shape}"
          f"   coarse range [{g['img_coarse'].min():.4f}, {g['img_coarse'].max():.4f}]")


In [ ]:
scene = "urban"
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(coarse[scene]["img_native"], cmap="gray")
axes[0].set_title(f"native, {coarse[scene]['img_native'].shape}")
axes[1].imshow(coarse[scene]["img_coarse"], cmap="gray")
axes[1].set_title(f"degraded ({gt.PIXEL_SIZE_COARSE_M:.0f} m), {coarse[scene]['img_coarse'].shape}")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.show()


## Step 6 -- PSF-aware reconstruction at REF_LEVEL

Inspect: how many of the reference's cells were actually reconstructed (`out_cell_ids` only guarantees a subset), and the build/solve timing.

In [ ]:
psf_results = {scene: gt.reconstruct_psf_aware(scene) for scene in gt.benchmark_coordinates}

for scene, d in psf_results.items():
    n_ref = len(refs[scene]["cell_ids_ref"])
    n_psf = len(d["cell_ids"])
    print(f"{scene:12s}  {n_psf:6d} / {n_ref:6d} reference cells reconstructed "
          f"({100*n_psf/n_ref:.1f}%)   build {float(d['build_time']):.2f}s  "
          f"solve {float(d['solve_time']):.2f}s")


## Baselines

Nearest / bilinear / bicubic (geometric, no spatial-response model) and Richardson-Lucy (two-stage deconvolution), all read out at the same reference cell centres as the PSF-aware result.

In [ ]:
classical_results = {
    (scene, method): gt.reconstruct_classical(scene, method=method)
    for scene in gt.benchmark_coordinates
    for method in gt.CLASSICAL_METHODS
}
rl_results = {scene: gt.reconstruct_richardson_lucy(scene) for scene in gt.benchmark_coordinates}

for scene in gt.benchmark_coordinates:
    counts = ", ".join(
        f"{gt.CLASSICAL_LABELS[m]}={len(classical_results[(scene, m)]['cell_ids'])}"
        for m in gt.CLASSICAL_METHODS
    )
    print(f"{scene:12s}  {counts}, RL={len(rl_results[scene]['cell_ids'])}")


## Step 7 -- alignment and metrics, per scene/method

This loop is the transparent, inspectable version of `run_all()` below: every `(scene, method)` pair, the number of cells actually compared after masking, and its metrics, printed as it goes.

In [ ]:
rows = []
for scene in gt.benchmark_coordinates:
    ref = refs[scene]
    g = coarse[scene]

    e, t, _ = gt.align_and_mask(psf_results[scene]["cell_ids"], psf_results[scene]["cell_data"], ref, g)
    rows.append(gt.compute_metrics(e, t, scene, "psf_aware"))

    for method in gt.CLASSICAL_METHODS:
        d = classical_results[(scene, method)]
        e, t, _ = gt.align_and_mask(d["cell_ids"], d["cell_data"], ref, g)
        rows.append(gt.compute_metrics(e, t, scene, f"classical_{method}"))

    d = rl_results[scene]
    e, t, _ = gt.align_and_mask(d["cell_ids"], d["cell_data"], ref, g)
    rows.append(gt.compute_metrics(e, t, scene, "richardson_lucy"))

metrics_df = pd.DataFrame(rows)
metrics_df


## Full run (cached, same result as the step-by-step loop above) + table + figures

In [ ]:
metrics_df = gt.run_all(force=False)
gt.format_table(metrics_df)


In [ ]:
gt.plot_scatter(force=False)


In [ ]:
gt.plot_panels(force=False)


In [ ]:
gt.plot_rmse_summary(metrics_df)


## Notes

- All parameters, function bodies, and the full design rationale (why this
  protocol, and why the two earlier ones were revised) live in
  `tests/real_groundtruth_common_tools.py`. This notebook is intentionally
  just calls.
- `force=True` on any `gt.build_*`/`gt.reconstruct_*` call recomputes and
  overwrites that step's cache; every cache filename already embeds
  `NATIVE_LEVEL`/`REF_LEVEL`/`BLOCK`/`RECON_FWHM_M`
  (`gt._cache_suffix()`), so changing those never silently reuses a stale
  result -- `force=False` is only a *speed* choice, not a correctness one.
- `refs`, `coarse`, `psf_results`, `classical_results`, `rl_results` are
  kept as plain dicts in this notebook's namespace specifically so you can
  poke at any intermediate array directly, e.g.
  `refs["urban"]["cell_data_ref"]` or `coarse["forest"]["img_coarse"]`.
